# 01 — SmarTRIZ QLoRA SFT Training

Trains Qwen2.5-7B-Instruct (or 14B) with QLoRA + CoT loss masking.

**Prerequisites:** Run Notebook 00 first to produce `training_dataset_clean.json` and `test_split.json`.

In [17]:
# ─── CONFIG — edit this cell only ────────────────────────
MODEL_SIZE = '7b'          # change to '14b' for 14B training
DRIVE_PATH = '/content/drive/MyDrive/LIFTUP/smartriz'
WANDB_PROJECT = 'smartriz-sft'

MODEL_NAME   = f'Qwen/Qwen2.5-{MODEL_SIZE.upper()}-Instruct'
DATASET_PATH = f'{DRIVE_PATH}/smartrizdata/training_dataset_clean.json'
OUTPUT_DIR   = f'{DRIVE_PATH}/smartriz/checkpoints/sft-{MODEL_SIZE}/'

LORA_R     = 16 if MODEL_SIZE == '7b' else 32
LORA_ALPHA = 32 if MODEL_SIZE == '7b' else 64
BATCH_SIZE = 4  if MODEL_SIZE == '7b' else 2
GRAD_ACCUM = 4  if MODEL_SIZE == '7b' else 8
MAX_SEQ_LEN   = 3072
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3
SAVE_STEPS    = 200
# ──────────────────────────────────────────────────────────

In [2]:
!pip uninstall -y bitsandbytes triton -q
!pip install -q transformers==4.45.2 peft==0.13.2 trl==0.11.4 accelerate==0.34.2 datasets==3.0.1 wandb tqdm
!pip install -q bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 169.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This beha

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ── Resume from checkpoint detection
from pathlib import Path

checkpoint_dirs = sorted(Path(OUTPUT_DIR).glob('checkpoint-*')) if Path(OUTPUT_DIR).exists() else []
RESUME_FROM = str(checkpoint_dirs[-1]) if checkpoint_dirs else None
if RESUME_FROM:
    print(f'Resuming from checkpoint: {RESUME_FROM}')
else:
    print('No checkpoint found — training from scratch')

No checkpoint found — training from scratch


In [5]:
import json

def load_training(path):
    with open(path) as f:
        data = json.load(f)
    return data['cases'] if isinstance(data, dict) else data

cases = load_training(DATASET_PATH)
print(f'Loaded {len(cases)} training examples')

Loaded 1489 training examples


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

SYSTEM_PROMPT = (
    'You are SmarTRIZ, an expert engineering innovation assistant. '
    'Solve technical problems using TRIZ methodology. Identify the '
    'technical contradiction, select inventive principles from the '
    'Altshuller matrix, reason step by step, and propose a solution.'
)

def format_example(case):
    cp = case['contradiction_pair']
    principles = ', '.join(case['inventive_principles'])
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': case['problem']},
        {'role': 'assistant', 'content': (
            f'<think>\n{case["reasoning_chain"]}\n</think>\n'
            f'Contradiction:\n\n'
            f'Improving: {cp["improving_parameter"]}\n'
            f'Worsening: {cp["worsening_parameter"]}\n\n'
            f'Inventive Principles: {principles}\n'
            f'Solution:\n{case["solution"]}'
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

sample = format_example(cases[0])
print('Sample formatted example (first 600 chars):')
print(sample[:600])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Sample formatted example (first 600 chars):
<|im_start|>system
You are SmarTRIZ, an expert engineering innovation assistant. Solve technical problems using TRIZ methodology. Identify the technical contradiction, select inventive principles from the Altshuller matrix, reason step by step, and propose a solution.<|im_end|>
<|im_start|>user
Design a battery pack enclosure for an electric vehicle that must withstand a 50 g frontal crash pulse while minimizing mass to preserve vehicle range. Current aluminum extrusions meet strength targets but add 18 kg over the baseline steel design, reducing range by an estimated 3.2%.<|im_end|>
<|im_star


In [7]:
# ── Custom CoT loss masking collator
# Masks labels for system+user turns and <think>/<\/think> tags themselves.
# Keeps loss on reasoning inside <think> and on structured output after <\/think>.
import torch
from dataclasses import dataclass
from typing import Any

@dataclass
class CotMaskingCollator:
    tokenizer: Any
    max_length: int = 3072

    def _tok(self, text):
        return self.tokenizer.encode(text, add_special_tokens=False)

    def _find_sublist(self, lst, sub):
        n = len(sub)
        for i in range(len(lst) - n + 1):
            if lst[i:i+n] == sub:
                return i
        return -1

    def __call__(self, features):
        im_start   = self._tok('<|im_start|>')
        im_end     = self._tok('<|im_end|>')
        user_tok   = self._tok('user')
        system_tok = self._tok('system')
        asst_tok   = self._tok('assistant')
        think_open  = self._tok('<think>')
        think_close = self._tok('</think>')

        batch_input_ids, batch_labels = [], []

        for f in features:
            ids = list(f['input_ids'][:self.max_length])
            labels = list(ids)
            i = 0
            while i < len(ids):
                if ids[i:i+len(im_start)] == im_start:
                    rs = i + len(im_start)
                    if ids[rs:rs+len(system_tok)] == system_tok or ids[rs:rs+len(user_tok)] == user_tok:
                        # Mask entire system/user turn
                        end = self._find_sublist(ids[i:], im_end)
                        if end != -1:
                            for j in range(i, i + end + len(im_end)):
                                if j < len(labels):
                                    labels[j] = -100
                    elif ids[rs:rs+len(asst_tok)] == asst_tok:
                        # Mask <|im_start|>assistant\n prefix
                        content_start = rs + len(asst_tok) + 1
                        for j in range(i, min(content_start, len(labels))):
                            labels[j] = -100
                        # Mask <think> tag itself
                        tp = self._find_sublist(ids[content_start:], think_open)
                        if tp != -1:
                            for j in range(content_start + tp, content_start + tp + len(think_open)):
                                if j < len(labels): labels[j] = -100
                        # Mask </think> tag itself
                        cp = self._find_sublist(ids[content_start:], think_close)
                        if cp != -1:
                            for j in range(content_start + cp, content_start + cp + len(think_close)):
                                if j < len(labels): labels[j] = -100
                i += 1

            batch_input_ids.append(ids)
            batch_labels.append(labels)

        max_len = max(len(x) for x in batch_input_ids)
        pad_id  = self.tokenizer.pad_token_id or 0
        input_ids = torch.tensor([x + [pad_id] * (max_len - len(x)) for x in batch_input_ids])
        labels_t  = torch.tensor([x + [-100]  * (max_len - len(x)) for x in batch_labels])
        attn_mask = (input_ids != pad_id).long()
        return {'input_ids': input_ids, 'attention_mask': attn_mask, 'labels': labels_t}

print('CotMaskingCollator defined')

CotMaskingCollator defined


In [8]:
# ── Verify masking on one example
sample_encoded = tokenizer(format_example(cases[0]), return_tensors='pt',
                           truncation=True, max_length=MAX_SEQ_LEN)
collator = CotMaskingCollator(tokenizer=tokenizer, max_length=MAX_SEQ_LEN)
batch = collator([{'input_ids': sample_encoded['input_ids'][0].tolist()}])

ids  = batch['input_ids'][0]
lbls = batch['labels'][0]

print('=== Masking verification (✓=loss computed, ✗=masked) — first 150 tokens ===')
for k in range(min(150, len(ids))):
    tok_str = tokenizer.decode([ids[k].item()], skip_special_tokens=False)
    mark = '✓' if lbls[k].item() != -100 else '✗'
    print(f'{mark}{tok_str!r}', end=' ')
print('\n...(truncated)')

=== Masking verification (✓=loss computed, ✗=masked) — first 150 tokens ===
✗'<|im_start|>' ✗'system' ✗'\n' ✗'You' ✗' are' ✗' Sm' ✗'ar' ✗'TR' ✗'IZ' ✗',' ✗' an' ✗' expert' ✗' engineering' ✗' innovation' ✗' assistant' ✗'.' ✗' Solve' ✗' technical' ✗' problems' ✗' using' ✗' TR' ✗'IZ' ✗' methodology' ✗'.' ✗' Identify' ✗' the' ✗' technical' ✗' contradiction' ✗',' ✗' select' ✗' inventive' ✗' principles' ✗' from' ✗' the' ✗' Alt' ✗'sh' ✗'ull' ✗'er' ✗' matrix' ✗',' ✗' reason' ✗' step' ✗' by' ✗' step' ✗',' ✗' and' ✗' propose' ✗' a' ✗' solution' ✗'.' ✗'<|im_end|>' ✓'\n' ✗'<|im_start|>' ✗'user' ✗'\n' ✗'Design' ✗' a' ✗' battery' ✗' pack' ✗' enclosure' ✗' for' ✗' an' ✗' electric' ✗' vehicle' ✗' that' ✗' must' ✗' withstand' ✗' a' ✗' ' ✗'5' ✗'0' ✗' g' ✗' frontal' ✗' crash' ✗' pulse' ✗' while' ✗' minimizing' ✗' mass' ✗' to' ✗' preserve' ✗' vehicle' ✗' range' ✗'.' ✗' Current' ✗' aluminum' ✗' extr' ✗'usions' ✗' meet' ✗' strength' ✗' targets' ✗' but' ✗' add' ✗' ' ✗'1' ✗'8' ✗' kg' ✗' over' ✗' the' ✗' baseli

In [9]:
# ── Cell 9 (fixed): Load model + LoRA with robust grad flow ──
import importlib.metadata as md
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    !nvidia-smi

# ---- Try 4-bit path first ----
use_4bit = False
bnb_config = None
try:
    bnb_ver = md.version("bitsandbytes")
    print("bitsandbytes:", bnb_ver)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    use_4bit = True
except Exception as e:
    print(f"[WARN] bitsandbytes not available, using bf16 fallback: {e}")

# ---- Load base model ----
if use_4bit:
    print("Loading model in 4-bit...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    # Required preparation for k-bit PEFT training
    model = prepare_model_for_kbit_training(model)
else:
    print("Loading model in bf16 fallback...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

model.config.use_cache = False

# ---- Gradient checkpointing / input grads safety ----
# Ensures backward graph is created correctly with checkpointing
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# ---- LoRA config ----
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Attach LoRA adapters
model = get_peft_model(model, lora_cfg)

# Optional but often helpful with checkpointing + k-bit
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# ---- Sanity checks ----
model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")

# Verify at least one trainable parameter exists
assert trainable > 0, "LoRA attach failed: no trainable params found"
print("Model + LoRA ready.")

CUDA available: True
Fri May  8 14:22:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+--------------------------

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
Trainable params: 40,370,176 / 4,393,342,464 (0.9189%)
Model + LoRA ready.


In [10]:
# ── Prepare HF Dataset with proper held-out eval split
import warnings
import json
from pathlib import Path
from datasets import Dataset as HFDataset
from tqdm.auto import tqdm

TEST_SPLIT_PATH = f'{DRIVE_PATH}data/test_split.json'

formatted = [{'text': format_example(c)} for c in tqdm(cases, desc='formatting train')]

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_SEQ_LEN, padding=False)

hf_train = HFDataset.from_list(formatted)
hf_train = hf_train.map(tokenize, batched=True, remove_columns=['text'])

# Load proper held-out eval set created by Notebook 00
if Path(TEST_SPLIT_PATH).exists():
    with open(TEST_SPLIT_PATH) as f:
        test_cases_raw = json.load(f)['cases']
    formatted_eval = [{'text': format_example(c)} for c in tqdm(test_cases_raw, desc='formatting eval')]
    hf_eval = HFDataset.from_list(formatted_eval)
    hf_eval = hf_eval.map(tokenize, batched=True, remove_columns=['text'])
    print(f'Loaded held-out eval set: {len(hf_eval)} examples from test_split.json')
else:
    warnings.warn(
        '\n\n⚠️  test_split.json not found — falling back to 5% of training data as eval.\n'
        '   Run Notebook 00 first to create a proper held-out eval set.\n',
        UserWarning, stacklevel=2
    )
    split = hf_train.train_test_split(test_size=0.05, seed=42)
    hf_train = split['train']
    hf_eval  = split['test']
    print(f'WARNING: Using internal eval split ({len(hf_eval)} examples) — NOT held-out data')

print(f'Train: {len(hf_train)} | Eval: {len(hf_eval)}')

formatting train:   0%|          | 0/1489 [00:00<?, ?it/s]

Map:   0%|          | 0/1489 [00:00<?, ? examples/s]

Train: 1414 | Eval: 75


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: UserWarning: 

⚠️  test_split.json not found — falling back to 5% of training data as eval.
   Run Notebook 00 first to create a proper held-out eval set.

  exec(code_obj, self.user_global_ns, self.user_ns)


In [11]:
# ── Train with SFTTrainer + custom collator
import wandb
from transformers import TrainingArguments
from trl import SFTTrainer

wandb.init(project=WANDB_PROJECT, name=f'sft-{MODEL_SIZE}', resume='allow')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    evaluation_strategy='steps',
    eval_steps=SAVE_STEPS,
    report_to='wandb',
    run_name=f'sft-{MODEL_SIZE}',
    optim='paged_adamw_8bit',
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_eval,
    data_collator=CotMaskingCollator(tokenizer=tokenizer, max_length=MAX_SEQ_LEN),
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LEN,
)

trainer.train(resume_from_checkpoint=RESUME_FROM)
print('Training complete')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sevketugurel (sevketugurel-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the 

Step,Training Loss,Validation Loss
200,1.178700,1.478267


Training complete


In [12]:
# ── Merge LoRA into base model and save
from pathlib import Path

merged_dir = OUTPUT_DIR + 'merged/'
if not Path(merged_dir + 'config.json').exists():
    print('Merging LoRA weights into base model...')
    model = model.merge_and_unload()
    model.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)
    print(f'Merged model saved to {merged_dir}')
else:
    print(f'Merged model already exists at {merged_dir} — skipping')

Merging LoRA weights into base model...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:336: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Merged model saved to /content/drive/MyDrive/LIFTUP/smartriz/checkpoints/sft-7b/merged/


In [18]:
# ── Quick eval: 5 held-out examples (expected vs generated)
import json
from transformers import pipeline

with open(f'{DRIVE_PATH}data/test_split.json') as f:
    test_cases = json.load(f)['cases'][:5]

gen_pipe = pipeline('text-generation', model=merged_dir, tokenizer=merged_dir,
                    device_map='auto', max_new_tokens=1024, do_sample=False)

for i, case in enumerate(test_cases):
    prompt = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user', 'content': case['problem']}],
        tokenize=False, add_generation_prompt=True
    )
    result = gen_pipe(prompt)[0]['generated_text']
    generated = result[len(prompt):].strip()
    print(f'=== Example {i+1} ===')
    print(f'EXPECTED SOLUTION (first 300 chars):\n{case["solution"][:300]}...')
    print(f'GENERATED (first 300 chars):\n{generated[:300]}...')
    print()

print('Ready for Notebook 02 (DPO training)')

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:623: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train 

=== Example 1 ===
EXPECTED SOLUTION (first 300 chars):
Triple-action facade retrofit: (1) Thermochromic paint on mullions (dark below 30°C, white above, solar reflectance shift from 0.3 to 0.8). (2) PCM-filled mullions (paraffin-based, 28°C melt point, 200 kJ/kg latent capacity) limit surface temperature to 32°C. (3) Interior ETFE film (200 µm, 95% VLT)...
GENERATED (first 300 chars):
Thank you for the detailed bug report. Let's address this issue step-by-step using the TRIZ methodology to find a solution that balances aesthetics, energy efficiency, and occupant comfort.

### Step 1: Identifying the Technical Contradiction

**Problem:** The glass facade provides good daylighting ...

=== Example 2 ===
EXPECTED SOLUTION (first 300 chars):
A roadside LiDAR detects an approaching vehicle; a modulated 589 nm amber laser projector paints the speed limit or curve warning as a large, low-scatter image onto the fog layer 10 m ahead of the car. Vehicle tracking adjust the projection in real tim

In [19]:
# ── Quick inference (SFT merged model) ──
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# merged_dir daha önce tanımlıysa onu kullanır; yoksa varsayılan yol
try:
    merged_dir
except NameError:
    merged_dir = "/content/drive/MyDrive/LIFTUP/smartriz/checkpoints/sft-7b/merged"

tokenizer = AutoTokenizer.from_pretrained(merged_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    merged_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

SYSTEM_PROMPT = (
    "You are SmarTRIZ, an expert engineering innovation assistant. "
    "Solve technical problems using TRIZ methodology. Identify the "
    "technical contradiction, select inventive principles from the "
    "Altshuller matrix, reason step by step, and propose a solution."
)

user_problem = "A bicycle chain stretches under repeated load and loses transmission efficiency. How can we improve durability without significantly increasing weight?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_problem},
]

prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=700,
        do_sample=False,          # deterministik
        temperature=0.0,
        top_p=1.0,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer = generated[len(prompt):].strip()

print("=== MODEL OUTPUT ===")
print(answer[:3000])

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


=== MODEL OUTPUT ===
repeated load and losing transmission efficiency while minimizing additional weight, we need to identify the technical contradiction and then apply TRIZ principles to find a solution.

### Step 1: Identify the Technical Contradiction

The problem involves two competing requirements:
- **Improving Durability**: This requires a material or design that can withstand more cycles of loading without stretching.
- **Minimizing Weight**: This requires a lighter material or design, which often means less robust materials.

These requirements are in conflict because typically, materials that are lighter are also less durable. Therefore, the technical contradiction is between "Durability" and "Weight."

### Step 2: Apply TRIZ Principles

We will use the Altshuller matrix to find an appropriate inventive principle. The matrix helps us match the technical contradiction with a suitable inventive principle.

For the contradiction between "Durability" and "Weight," the Altshuller 